# Introduction
This colab worksheet provides a starting point for Task 2 (the computer vision task).

# Data Loading

In [ ]:
# Download the data stored in a zipped numpy array from one of these two locations
# If you're running all your experiments
# on a machine at home rather than using colab, then make sure you save it
# rather than repeatedly downloading it.

!wget "https://sussex.box.com/shared/static/wrlmcs47dstkmfz2eckgg5d2l0liusr5" -O face_alignment_training_images.npz

# The test images (without points)
!wget "https://sussex.box.com/shared/static/8n5ofhb9so0ia85g4bbcmnt3h6z53m9a" -O face_alignment_test_images.npz

# Check the data downloaded correctly
If any of these assertions fail, redownload the data

In [ ]:
def confirm_checksum(filename, true_checksum):
  import subprocess
  checksum = subprocess.check_output(['shasum',filename]).decode('utf-8')
  print (checksum)
  assert checksum.split(' ')[0] == true_checksum, 'Checksum does not match for ' + filename + ' redownload the data.'

confirm_checksum('face_alignment_training_images.npz', 'affc56216e34c827f10681a14c602981c06ddeb3')
confirm_checksum('face_alignment_test_images.npz', 'e480535442512b8175ae110d2d3e507378cabbdb')



# Load the data

In [ ]:
import numpy as np


# Load the data using np.load
data = np.load('face_alignment_training_images.npz', allow_pickle=True)

# Extract the images
images = data['images']
# and the data points
pts = data['points']

print(images.shape, pts.shape)

test_data = np.load('face_alignment_test_images.npz', allow_pickle=True)
test_images = test_data['images']
print(test_images.shape)

# Data Visualisation
Here's an example of how to display the images and their points

In [ ]:

def visualise_pts(img, pts):
  import matplotlib.pyplot as plt
  plt.imshow(img)
  plt.plot(pts[:, 0], pts[:, 1], '+r')
  plt.show()

for i in range(3):
  idx = np.random.randint(0, images.shape[0])
  visualise_pts(images[idx, ...], pts[idx, ...])

# Calculating Prediction Error and exporting results

In [ ]:
def euclid_dist(pred_pts, gt_pts):
  """
  Calculate the euclidean distance between pairs of points
  :param pred_pts: The predicted points
  :param gt_pts: The ground truth points
  :return: An array of shape (no_points,) containing the distance of each predicted point from the ground truth
  """
  import numpy as np
  pred_pts = np.reshape(pred_pts, (-1, 2))
  gt_pts = np.reshape(gt_pts, (-1, 2))
  return np.sqrt(np.sum(np.square(pred_pts - gt_pts), axis=-1))


In [ ]:
def save_as_csv(points, location = '.'):
    """
    Save the points out as a .csv file
    :param points: numpy array of shape (no_test_images, no_points, 2) to be saved
    :param location: Directory to save results.csv in. Default to current working directory
    """
    assert points.shape[0]==554, 'wrong number of image points, should be 554 test images'
    assert np.prod(points.shape[1:])==5*2, 'wrong number of points provided. There should be 5 points with 2 values (x,y) per point'
    np.savetxt(location + '/results_task2.csv', np.reshape(points, (points.shape[0], -1)), delimiter=',')


In [ ]:
import joblib  # For saving and loading machine learning models
import cv2  # OpenCV for image processing tasks
import pandas as pd  # For working with structured data (DataFrames, CSV, etc.)
import matplotlib.pyplot as plt  # To plot graphs and visualizations
import numpy as np  # Fundamental numerical operations and array handling
from sklearn.model_selection import train_test_split  # Splits data into training and testing subsets
from sklearn.preprocessing import StandardScaler  # Standardizes features by removing the mean and scaling to unit variance
from sklearn.linear_model import Ridge  # Ridge regression model for regularized linear regression
from sklearn.metrics import mean_absolute_error  # Metric to evaluate prediction error by average absolute difference


In [ ]:
def generate_dense_keypoints(dimensions, spacing=32):
    """
    Creates a list of keypoints in a uniform grid layout across an image.

    Parameters:
    - dimensions: Tuple (height, width) of the image.
    - spacing: Distance (in pixels) between keypoints along both axes.

    Returns:
    - A list of cv2.KeyPoint objects laid out in a grid pattern.
    """
    height, width = dimensions
    points = [
        cv2.KeyPoint(float(x), float(y), spacing)
        for y in range(spacing // 2, height, spacing)
        for x in range(spacing // 2, width, spacing)
    ]
    return points

In [ ]:
def sift_grid_feature_extraction(batch_images, grid_spacing=32):
    """
    Extracts SIFT descriptors from a list of images using a fixed spatial grid of keypoints.

    Parameters:
    - batch_images: A numpy array of shape (N, H, W, 3) representing a batch of RGB images.
    - grid_spacing: Distance between grid points for keypoint placement.

    Returns:
    - A numpy array of flattened SIFT descriptors for each image.
    - The keypoints used for descriptor extraction.
    """
    sift_engine = cv2.SIFT_create()
    feature_matrix = []

    # Generate keypoints template from the shape of a single image
    img_height, img_width = batch_images.shape[1:3]
    grid_keypoints = generate_dense_keypoints((img_height, img_width), spacing=grid_spacing)

    for image in batch_images:
        # Normalize and convert to uint8 if necessary
        if image.dtype != np.uint8:
            image = (np.clip(image, 0, 1) * 255).astype(np.uint8)

        grayscale_img = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        _, descriptor_set = sift_engine.compute(grayscale_img, grid_keypoints)

        # Handle images where SIFT fails to produce descriptors
        if descriptor_set is None:
            descriptor_set = np.zeros((len(grid_keypoints), 128), dtype=np.float32)

        feature_matrix.append(descriptor_set.flatten())

    return np.array(feature_matrix), grid_keypoints


In [ ]:
def visualize_prediction_errors(error_values):
    """
    Visualize prediction error distribution using KDE, violin plot, and ECDF.

    Parameters:
    - error_values: Array-like, containing prediction errors.
    """
    import seaborn as sns
    from statsmodels.distributions.empirical_distribution import ECDF

    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

    # Kernel Density Estimate (smoothed distribution)
    sns.histplot(error_values, fill=True, ax=axes[0], color='skyblue')
    axes[0].set_title("Smoothed Error Distribution")
    axes[0].set_xlabel("Error")

    # Violin plot for spread and central tendency
    sns.violinplot(y=error_values, ax=axes[1], color='lightgreen')
    axes[1].set_title("Error Spread (Violin Plot)")
    axes[1].set_ylabel("Error")

    # Empirical CDF using statsmodels
    ecdf = ECDF(error_values)
    axes[2].plot(ecdf.x, ecdf.y, color='coral')
    axes[2].set_title("Empirical Cumulative Distribution")
    axes[2].set_xlabel("Error")
    axes[2].set_ylabel("Cumulative Probability")

    plt.suptitle("Error Analysis Overview", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
def prepare_dataset_with_sift(images, target_points, test_ratio=0.2, random_seed=42, step_size=32):
    """
    Full preprocessing pipeline:
    - Extracts SIFT descriptors on a grid.
    - Flattens target points.
    - Splits into training and validation sets.
    - Applies standard scaling to features and targets.

    Parameters:
    - images: Numpy array of RGB images (N, H, W, 3)
    - target_points: Numpy array of keypoint coordinates (N, num_points, 2)
    - test_ratio: Fraction of data reserved for validation
    - random_seed: Seed for reproducibility
    - step_size: Grid step size for SIFT keypoint layout

    Returns:
    - Dictionary containing scaled train/val sets and fitted scalers
    """
    print("Starting SIFT grid descriptor extraction...")
    feature_matrix, _ = sift_grid_feature_extraction(images, grid_spacing=step_size)


    print("Reshaping target coordinates...")
    target_flat = target_points.reshape(target_points.shape[0], -1)

    print("Splitting data into training and validation sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        feature_matrix, target_flat, test_size=test_ratio, random_state=random_seed
    )

    print("Applying standard scaling to features and targets...")
    scaler_features = StandardScaler()
    scaler_targets = StandardScaler()

    X_train_scaled = scaler_features.fit_transform(X_train)
    X_test_scaled = scaler_features.transform(X_test)

    y_train_scaled = scaler_targets.fit_transform(y_train)
    y_test_scaled = scaler_targets.transform(y_test)

    return {
        "X_train": X_train_scaled,
        "X_val": X_test_scaled,
        "y_train": y_train_scaled,
        "y_val": y_test_scaled,
        "scaler_X": scaler_features,
        "scaler_y": scaler_targets
    }


In [ ]:
def train_and_evaluate_ridge(features_train, features_val, targets_train, targets_val, scaler_y, model_label="RidgeRegressor"):
    """
    Trains a Ridge regression model and evaluates performance using Euclidean error metrics.

    Parameters:
    - features_train: Scaled training features
    - features_val: Scaled validation features
    - targets_train: Scaled training targets
    - targets_val: Scaled validation targets
    - scaler_y: Target scaler (used to inverse transform predictions)
    - model_label: String identifier for the model

    Returns:
    - Dictionary with error statistics
    """
    print(f"Fitting {model_label}...")
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(features_train, targets_train)

    # Predict and rescale outputs
    predicted_scaled = ridge_model.predict(features_val)
    predicted = scaler_y.inverse_transform(predicted_scaled)

    # Calculate Euclidean distance errors
    flat_errors = []
    for truth, pred in zip(targets_val, predicted):
        gt_points = truth.reshape(-1, 2)
        pred_points = pred.reshape(-1, 2)
        dists = np.linalg.norm(gt_points - pred_points, axis=1)
        flat_errors.extend(dists)

    # Summarize results
    summary = {
        "model_name": model_label,
        "avg_error": np.mean(flat_errors),
        "median_error": np.median(flat_errors),
        "q3_error": np.percentile(flat_errors, 75)
    }

    # Visual diagnostics
    visualize_prediction_errors(flat_errors)

    return ridge_model, summary


In [ ]:
# Run full preprocessing pipeline
data_bundle = prepare_dataset_with_sift(images, pts)

# Unpack scaled datasets
X_train_scaled = data_bundle["X_train"]
X_val_scaled = data_bundle["X_val"]
y_train_scaled = data_bundle["y_train"]
y_val = data_bundle["y_val"]
scaler_y = data_bundle["scaler_y"]
scaler_X = data_bundle["scaler_X"]


In [ ]:
# Run the training and evaluation
ridge_model, ridge_summary = train_and_evaluate_ridge(
    features_train=X_train_scaled,
    features_val=X_val_scaled,
    targets_train=y_train_scaled,
    targets_val=scaler_y.transform(y_val),  # Ensure scaled input
    scaler_y=scaler_y
)

# Append to result log
results = [ridge_summary]


In [ ]:
class StageWiseRegressor:
    def __init__(self, base_estimator, stages=3, estimator_params=None):
        """
        A custom regressor that refines predictions over multiple stages using residual feedback.

        Parameters:
        - base_estimator: a regressor class (e.g., Ridge)
        - stages: number of refinement stages
        - estimator_params: parameters to pass to each base model
        """
        self.stages = stages
        self.estimator_params = estimator_params or {}
        self.pipeline = [base_estimator(**self.estimator_params) for _ in range(stages)]

    def fit(self, X, y):
        """
        Fit all stages of the model sequentially using residual correction.
        """
        prediction = np.zeros_like(y)
        for i, model in enumerate(self.pipeline):
            residual = y - prediction
            model.fit(X, residual)
            prediction += model.predict(X)

    def predict(self, X):
        """
        Aggregate predictions from all stages.
        """
        output = np.zeros((X.shape[0], self.pipeline[0].coef_.shape[0]))
        for model in self.pipeline:
            output += model.predict(X)
        return output


In [ ]:
# Initialize and train custom cascaded model
stagewise = StageWiseRegressor(base_estimator=Ridge, stages=3, estimator_params={"alpha": 1.0})
stagewise.fit(X_train_scaled, y_train_scaled)

# Predict and rescale
stagewise_preds_scaled = stagewise.predict(X_val_scaled)
stagewise_preds = scaler_y.inverse_transform(stagewise_preds_scaled)

# Compute per-point Euclidean errors
stagewise_errors = []
for pred, true in zip(stagewise_preds, y_val):
    dist = np.linalg.norm(pred.reshape(-1, 2) - true.reshape(-1, 2), axis=1)
    stagewise_errors.extend(dist)

# Record performance
results.append({
    "model_name": "StageWise Ridge Ensemble",
    "avg_error": np.mean(stagewise_errors),
    "median_error": np.median(stagewise_errors),
    "q3_error": np.percentile(stagewise_errors, 75)
})

# Visualize the error distribution
visualize_prediction_errors(stagewise_errors)


In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
def predict_landmarks_from_images(model, test_imgs, x_scaler, y_scaler):
    """
    Extract SIFT features from test images and predict landmark positions.

    Parameters:
    - model: trained regression model
    - test_imgs: array of test images (RGB)
    - x_scaler: fitted feature scaler
    - y_scaler: fitted target scaler

    Returns:
    - np.ndarray: predicted landmarks in original scale
    """
    print("[INFO] Generating feature descriptors for test set...")
    descriptors, _ = sift_grid_feature_extraction(test_imgs)

    print("[INFO] Normalizing test features...")
    X_scaled = x_scaler.transform(descriptors)

    print("[INFO] Predicting landmark coordinates...")
    y_pred_scaled = model.predict(X_scaled)
    y_pred_original = y_scaler.inverse_transform(y_pred_scaled)

    return y_pred_original


In [ ]:
# Run prediction pipeline
final_predictions = predict_landmarks_from_images(
    model=stagewise,
    test_imgs=test_images,
    x_scaler=scaler_X,
    y_scaler=scaler_y
)

# Save results
output_path = "landmarks_output.csv"
save_as_csv(pd.DataFrame(final_predictions))
print(f"[SUCCESS] Landmark predictions saved to: {output_path}")
